In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import re

# 1. Muat data hasil scraping dari Tugas 1
df = pd.read_csv('/content/drive/MyDrive/Muhammad Yahya Ayyasy_Tugas 21 Agustus 2026/scraped_playstore.csv')
print('Dimensi data awal:', df.shape)
df.head()

Dimensi data awal: (11000, 4)


,id,teks,tanggal,sumber
0,83256b2d-b2a9-483f-8abf-c0d3395fbe9a,"Suka betul deh ya, program & internalnya lebih...",2026-08-23 03:14:20,playstore
1,00485803-201e-4753-8a85-a8f7e080e781,"Suka sekali dana, program & internalnya lebih ...",2026-08-15 19:13:08,playstore
2,703e299d-a065-4520-bdc2-cdaf7072d29b,"akun sudah terverifikasi atau sudah premium, t...",2026-08-01 08:02:57,playstore
3,2420d506-f666-4710-9276-681d0ae270c4,"Help Center hanya dibantu oleh AI, sangat tida...",2026-08-10 05:17:00,playstore
4,5e17c24a-47d9-491d-95cd-72e76004a3ed,overall udh cukup bagus sih. tapi kenapa ya ba...,2026-08-16 14:52:08,playstore


In [ ]:
# 2. Kamus Normalisasi Slang (Bahasa Gaul e-wallet)
# Berfungsi mengubah kata singkatan menjadi kata baku agar terbaca oleh Lexicon
kamus_slang = {
    'apk': 'aplikasi', 'app': 'aplikasi', 'tf': 'transfer', 'trf': 'transfer',
    'wd': 'tarik', 'cs': 'layanan', 'bot': 'robot', 'bgs': 'bagus',
    'bgus': 'bagus', 'bgt': 'sangat', 'banget': 'sangat', 'ga': 'tidak',
    'gak': 'tidak', 'gk': 'tidak', 'engga': 'tidak', 'nggak': 'tidak',
    'klo': 'kalau', 'kalo': 'kalau', 'tp': 'tetapi', 'udh': 'sudah',
    'sdh': 'sudah', 'udah': 'sudah', 'blm': 'belum', 'blom': 'belum',
    'krn': 'karena', 'karna': 'karena', 'jd': 'jadi', 'jdi': 'jadi',
    'nyangkut': 'gagal', 'lemot': 'lambat', 'lmot': 'lambat'
}

# 3. Kamus Lexicon yang Diperluas (Khusus Industri Dompet Digital)
positif = [
    'bagus', 'mantap', 'puas', 'cepat', 'rekomen', 'recommended', 'top', 'oke',
    'baik', 'memuaskan', 'cocok', 'nyaman', 'murah', 'berkualitas', 'amanah',
    'ramah', 'rapi', 'sesuai', 'original', 'asli', 'suka', 'love', 'best',
    'hebat', 'kerennn', 'keren', 'terbaik', 'sukses', 'lancar', 'aman',
    'praktis', 'gampang', 'mudah', 'membantu', 'terpercaya', 'cashback',
    'promo', 'gratis', 'bintang 5', 'the best', 'canggih', 'berguna', 'jos',
    'untung', 'sempurna', 'bermanfaat', 'luar biasa', 'terbantu'
]

negatif = [
    'buruk', 'jelek', 'lambat', 'lama', 'kecewa', 'rusak', 'mengecewakan',
    'palsu', 'sobek', 'cacat', 'mahal', 'busuk', 'parah', 'kapok', 'penipu',
    'tipu', 'bohong', 'kotor', 'bau', 'tidak sesuai', 'sampah', 'norak',
    'error', 'hilang', 'najis', 'gagal', 'potong', 'rugi', 'susah', 'ribet',
    'maintenance', 'tutup', 'blokir', 'kembalikan', 'balikin', 'pencuri',
    'maling', 'lelet', 'update', 'upgrade', 'login', 'otp', 'verifikasi',
    'bug', 'bugs', 'kendala', 'macet', 'nihil', 'ngelag', 'lag', 'force close',
    'bekukan', 'dibekukan', 'penipuan', 'scam', 'lapor', 'bermasalah', 'perbaiki'
]

# 4. Kata Negasi untuk Pembalikan Sentimen (Negation Handling)
kata_negasi = ['tidak', 'bukan', 'jangan', 'kurang', 'anti', 'belum']

print('Jumlah kata positif:', len(positif))
print('Jumlah kata negatif:', len(negatif))


Jumlah kata positif: 48
Jumlah kata negatif: 58


In [ ]:
# 5. Fungsi Normalisasi Teks Dasar
def normalisasi_teks(teks):
    if not isinstance(teks, str):
        return ''
    teks = teks.lower()
    teks = re.sub(r'[^a-z0-9\s]', ' ', teks)
    kata_kata = teks.split()
    kata_normal = [kamus_slang.get(k, k) for k in kata_kata]
    return ' '.join(kata_normal)

In [ ]:
# 6. Fungsi Auto Labeling
def auto_label(teks):
    if not teks.strip():
        return 'netral'

    kata_kata = teks.split()
    skor_pos = 0
    skor_neg = 0

    for i, kata in enumerate(kata_kata):
        is_negated = False
        if i > 0 and kata_kata[i-1] in kata_negasi:
            is_negated = True

        if kata in positif:
            if is_negated: skor_neg += 1
            else: skor_pos += 1
        elif kata in negatif:
            if is_negated: skor_pos += 1
            else: skor_neg += 1

    if skor_pos > skor_neg: return 'positif'
    elif skor_neg > skor_pos: return 'negatif'
    else: return 'netral'

In [ ]:
# 7. Eksekusi
df['teks_normal'] = df['teks'].apply(normalisasi_teks)
df['sentimen_auto'] = df['teks_normal'].apply(auto_label)

print('\nDistribusi sentimen otomatis (Sebelum koreksi):')
print(df['sentimen_auto'].value_counts())


Distribusi sentimen otomatis (Sebelum koreksi):
sentimen_auto
negatif    5274
netral     3054
positif    2672
Name: count, dtype: int64


In [ ]:
sample = df.head(50).copy()
sample.to_csv('untuk_koreksi_manual.csv', index=False)
print("\nFile 'untuk_koreksi_manual.csv' berhasil dibuat untuk dikumpulkan!")


File 'untuk_koreksi_manual.csv' berhasil dibuat untuk dikumpulkan!


In [ ]:
# [CARA 2] Koreksi inline di notebook (untuk demo/praktis)
# Koreksi data DANA yang bias secara konteks bisnis
sample.loc[16, 'sentimen_auto'] = 'negatif'   # Asli Netral, pdhl transfer gagal
sample.loc[17, 'sentimen_auto'] = 'negatif'   # Asli Positif, pdhl delay & gagal
sample.loc[34, 'sentimen_auto'] = 'negatif'   # Asli Netral, pdhl keamananan lemah
sample.loc[42, 'sentimen_auto'] = 'negatif'   # Asli Netral, pdhl sarkas akun terkunci
sample.loc[47, 'sentimen_auto'] = 'negatif'   # Asli Netral, pdhl ngebug

In [ ]:
sample.to_csv('sudah_dikoreksi.csv', index=False)
print("File 'sudah_dikoreksi.csv' berhasil dibuat untuk dikumpulkan!")

File 'sudah_dikoreksi.csv' berhasil dibuat untuk dikumpulkan!


In [ ]:
df.loc[:49, 'sentimen_auto'] = sample['sentimen_auto'].values
df = df.rename(columns={'sentimen_auto': 'sentimen'})

In [ ]:
df_2kelas = df[df['sentimen'].isin(['positif', 'negatif'])].reset_index(drop=True)
df_2kelas.to_csv('data_labeled.csv', index=False)

In [ ]:
print('\nDistribusi sentimen final (Data Bersih 2 Kelas):')
print(df_2kelas['sentimen'].value_counts())


Distribusi sentimen final (Data Bersih 2 Kelas):
sentimen
negatif    5277
positif    2671
Name: count, dtype: int64
